In [ ]:
import os
import re
import json
import pandas as pd
import pdfplumber
import PyMuPDF
from pathlib import Path
from bs4 import BeautifulSoup
from transformers import pipeline
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer
import time
def setup_directories():
    """Sets up the required directories for the project."""
    dirs = ['data/raw_html', 'data/processed_data', 'data/nlp_results']
    for dir_path in dirs:
        Path(dir_path).mkdir(parents=True, exist_ok=True)
    print("✅ Directories are ready.")
print("📦 Loading smart NLP models...")
SUMMARIZER = None
KEYWORD_MODEL = None
EMBEDDING_MODEL = None
try:
    SUMMARIZER = pipeline("summarization", model="t5-base")
    print("✅ Summarization model loaded.")
    KEYWORD_MODEL = KeyBERT('all-mpnet-base-v2')
    EMBEDDING_MODEL = SentenceTransformer('all-mpnet-base-v2')
    print("✅ Keyword and embedding models loaded.")
except Exception as e:
    print(f"❌ Error loading models: {e}. Check your internet connection.")
def extract_text_from_pdf(pdf_path):
    """Extracts text from a PDF file using multiple methods."""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = "".join(page.extract_text() or "" for page in pdf.pages)
        if text.strip():
            return text, "pdfplumber"
        
        doc = PyMuPDF.open(pdf_path)
        text = "".join(page.get_text() for page in doc)
        doc.close()
        return text, "pymupdf"
    except Exception as e:
        return None, f"error: {str(e)}"
def extract_text_from_html(html_path):
    """Extracts text from an HTML file."""
    try:
        with open(html_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        soup = BeautifulSoup(content, 'html.parser')
        
        for script in soup(["script", "style"]):
            script.decompose()
        
        title = (soup.find('h1') or soup.find('title') or '').get_text().strip()
        abstract_section = soup.find('div', class_='abstract') or soup.find('section', class_='abstract')
        abstract = abstract_section.get_text().strip() if abstract_section else ""
        
        main_content = soup.find('div', class_='pmc-articleinfo') or soup.find('article') or soup.find('main')
        full_text = main_content.get_text() if main_content else soup.get_text()
        
        full_text = re.sub(r'\s+', ' ', full_text).strip()
        abstract = re.sub(r'\s+', ' ', abstract).strip()
        
        return {
            'title': title,
            'abstract': abstract,
            'full_text': full_text
        }, "success"
    except Exception as e:
        return None, f"error: {str(e)}"
def detect_sections(text):
    """Detects article sections using a rule-based approach."""
    sections = {
        'introduction': '', 'methods': '', 'results': '',
        'discussion': '', 'conclusion': ''
    }
    
    patterns = {
        "introduction": r'^(introduction|background)',
        "methods": r'^(methods|experimental|materials)',
        "results": r'^(results|findings)',
        "conclusion": r'^(conclusion|discussion|summary)'
    }
    
    paragraphs = re.split(r'\n\s*\n', text)
    current_section = "unknown"
    
    for p in paragraphs:
        p = p.strip()
        if not p:
            continue
        
        is_heading_found = False
        for section_name, pattern in patterns.items():
            if re.match(pattern, p, re.IGNORECASE):
                current_section = section_name
                is_heading_found = True
                break
        
        sections[current_section] = sections.get(current_section, '') + p + "\n\n"
            
    return sections
def extract_keywords_and_topics(text):
    """Extracts keywords and topics using a semantic approach with KeyBERT."""
    if KEYWORD_MODEL is None:
        return [], []
    
    keywords = KEYWORD_MODEL.extract_keywords(text, keyphrase_ngram_range=(1, 3), top_n=10)
    extracted_keywords = [kw[0] for kw in keywords]
    topics = extracted_keywords[:3]
    
    return extracted_keywords, topics
def generate_summaries(text):
    """Generates three types of summaries from the text using a T5 model."""
    if SUMMARIZER is None:
        return {"short": "", "medium": "", "long": ""}
        
    tldr = SUMMARIZER(text, max_length=50, min_length=20, do_sample=False)[0]['summary_text']
    medium = SUMMARIZER(text, max_length=150, min_length=100, do_sample=False)[0]['summary_text']
    long = SUMMARIZER(text, max_length=400, min_length=300, do_sample=False)[0]['summary_text']
    return {"short": tldr, "medium": medium, "long": long}
def create_embeddings(text):
    """Creates a vector embedding of the article text for semantic search."""
    if EMBEDDING_MODEL is None:
        return []
    
    embedding = EMBEDDING_MODEL.encode(text)
    
    return embedding.tolist()
def process_all_articles(csv_path, output_dir):
    """Orchestrates the entire article processing pipeline."""
    setup_directories()
    
    try:
        download_results = pd.read_csv(csv_path)
    except FileNotFoundError:
        print("❌ 'download_results.csv' not found. Please run the previous step.")
        return
        
    os.makedirs(output_dir, exist_ok=True)
    processed_articles = []
    
    for idx, row in download_results.iterrows():
        article_id = row['article_id']
        file_path = row['file_path']
        title = row['title']
        
        if pd.isna(file_path) or not Path(file_path).exists():
            continue
        print(f"\n🔄 Processing article {article_id}...")
        
        text = ""
        if file_path.endswith('.pdf'):
                    text, _ = extract_text_from_pdf(file_path)
    elif file_path.endswith('.html'):
        data, _ = extract_text_from_html(file_path)
            if data and 'full_text' in data:
                text = data['full_text']
        
        if not text:
            print(f"❌ No text extracted from {file_path}")
            continue
        
        sections = detect_sections(text)
        keywords, topics = extract_keywords_and_topics(text)
        summaries = generate_summaries(text)
        embeddings = create_embeddings(text)
        
        
        article_data = {
            "article_id": article_id,
            "title": title,
            "sections": sections,
            "nlp_analysis": {
                "keywords": keywords,
                "topics": topics,
                "summaries": summaries
            },
            "embeddings": embeddings,
            "full_text": text[:20000] 
        }
        processed_articles.append(article_data)
        
        print(f"✅ Article {article_id} processed successfully.")
        
    
    final_json_path = os.path.join(output_dir, "articles_nlp_processed.json")
    with open(final_json_path, 'w', encoding='utf-8') as f:
        json.dump(processed_articles, f, ensure_ascii=False, indent=2)
    print(f"\n🎉 Processing complete! Final JSON saved to {final_json_path}")
if name == "main":
    csv_input_path = "data/processed_data/download_results.csv" 
    output_directory = "data/nlp_results" 
    process_all_articles(csv_input_path, output_directory) 

In [3]:
import pandas as pd
import requests
from pathlib import Path
import time
import re
from bs4 import BeautifulSoup
import os
from datetime import datetime
import json

def setup_directories():
    """ساخت پوشه‌های مورد نیاز"""
    dirs = ['data/raw_html', 'data/processed_data']
    for dir_path in dirs:
        Path(dir_path).mkdir(parents=True, exist_ok=True)
    print("✅ پوشه‌ها ساخته شدند")

def download_html_from_pmc(url, article_id, max_retries=3):
    """دانلود HTML از PMC"""
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Connection': 'keep-alive',
    }
    
    session = requests.Session()
    session.headers.update(headers)
    
    for attempt in range(max_retries):
        try:
            print(f"   📡 تلاش دانلود {attempt + 1}/{max_retries}")
            
            response = session.get(url, timeout=30)
            response.raise_for_status()
            
            if response.status_code == 200:
                # ذخیره HTML
                html_path = f"data/raw_html/article_{article_id:03d}.html"
                with open(html_path, 'w', encoding='utf-8') as f:
                    f.write(response.text)
                
                return html_path, "success"
            
        except requests.exceptions.ConnectionError as e:
            print(f"   ❌ خطای اتصال: منتظر {(attempt + 1) * 5} ثانیه...")
            if attempt < max_retries - 1:
                time.sleep((attempt + 1) * 5)
            continue
            
        except requests.exceptions.Timeout:
            print(f"   ⏰ خطای timeout")
            if attempt < max_retries - 1:
                time.sleep(5)
            continue
            
        except Exception as e:
            print(f"   ❌ خطا: {str(e)[:100]}")
            if attempt < max_retries - 1:
                time.sleep(3)
            continue
    
    return None, f"failed_after_{max_retries}_attempts"

def extract_sections_from_html(html_content):
    """استخراج بخش‌های مختلف مقاله از HTML"""
    
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # حذف elements غیرضروری
    for element in soup(['script', 'style', 'nav', 'header', 'footer', 'aside']):
        element.decompose()
    
    sections = {
        'title': '',
        'abstract': '',
        'introduction': '',
        'methods': '',
        'results': '',
        'discussion': '',
        'conclusion': '',
        'references': '',
        'full_text': '',
        'authors': ''
    }
    
    # استخراج عنوان
    title_selectors = [
        'h1.content-title',
        'h1.article-title', 
        '.article-title',
        'h1',
        '.title-group h1',
        '.article-meta h1'
    ]
    
    for selector in title_selectors:
        title_elem = soup.select_one(selector)
        if title_elem:
            sections['title'] = title_elem.get_text().strip()
            break
    
    # استخراج نویسندگان
    author_selectors = [
        '.contrib-group .contrib',
        '.authors .author',
        '.author-list .author',
        '.contrib'
    ]
    
    authors = []
    for selector in author_selectors:
        author_elems = soup.select(selector)
        for elem in author_elems:
            author_name = elem.get_text().strip()
            if author_name and len(author_name) > 2:
                authors.append(author_name)
        if authors:
            break
    
    sections['authors'] = ', '.join(authors[:10])  # حداکثر 10 نویسنده
    
    # استخراج Abstract
    abstract_selectors = [
        '.abstract',
        '#abstract',
        '.section.abstract',
        'section[id*="abstract"]',
        '.abs'
    ]
    
    for selector in abstract_selectors:
        abstract_elem = soup.select_one(selector)
        if abstract_elem:
            abstract_text = abstract_elem.get_text().strip()
            # حذف کلمه "Abstract" از ابتدا
            abstract_text = re.sub(r'^(abstract|ABSTRACT)\s*', '', abstract_text, flags=re.IGNORECASE)
            sections['abstract'] = ' '.join(abstract_text.split())
            break
    
    # استخراج بخش‌های اصلی مقاله
    section_patterns = {
        'introduction': [
            r'introduction',
            r'background'
        ],
        'methods': [
            r'methods',
            r'materials?\s+and\s+methods',
            r'methodology',
            r'experimental\s+procedures?'
        ],
        'results': [
            r'results',
            r'findings'
        ],
        'discussion': [
            r'discussion',
            r'results\s+and\s+discussion'
        ],
        'conclusion': [
            r'conclusions?',
            r'summary',
            r'concluding\s+remarks?'
        ]
    }
    
    # جستجو در headings
    headings = soup.find_all(['h1', 'h2', 'h3', 'h4'], string=re.compile(r'.+'))
    
    for section_name, patterns in section_patterns.items():
        for heading in headings:
            heading_text = heading.get_text().strip().lower()
            
            for pattern in patterns:
                if re.search(pattern, heading_text, re.IGNORECASE):
                    # پیدا کردن محتوای این بخش
                    section_content = []
                    current = heading.next_sibling
                    
                    while current:
                        if hasattr(current, 'name'):
                            if current.name in ['h1', 'h2', 'h3', 'h4']:
                                # رسیدیم به heading بعدی
                                break
                            elif current.name == 'p':
                                section_content.append(current.get_text().strip())
                            elif current.name == 'div':
                                section_content.append(current.get_text().strip())
                        current = current.next_sibling
                    
                    if section_content:
                        sections[section_name] = ' '.join(section_content)
                        break
                    
            if sections[section_name]:  # اگر پیدا کردیم، بقیه patterns رو چک نکن
                break
    
    # استخراج References
    ref_selectors = [
        '.ref-list',
        '#references',
        '.references',
        '.bibliography'
    ]
    
    for selector in ref_selectors:
        ref_elem = soup.select_one(selector)
        if ref_elem:
            refs = ref_elem.find_all(['li', 'p', 'div'])
            ref_texts = [ref.get_text().strip() for ref in refs[:10]]  # حداکثر 10 reference
            sections['references'] = ' | '.join(ref_texts)
            break
    
    # استخراج کل متن مقاله
    main_content_selectors = [
        '.pmc-articleinfo',
        '.article-content',
        '.main-content',
        'main',
        '.content'
    ]
    
    full_text = ""
    for selector in main_content_selectors:
        main_elem = soup.select_one(selector)
        if main_elem:
            full_text = main_elem.get_text()
            break
    
    if not full_text:
        # اگر main content پیدا نشد، کل body رو بگیر
        body = soup.find('body')
        if body:
            full_text = body.get_text()
    
    # تمیز کردن متن کامل
    full_text = ' '.join(full_text.split())
    sections['full_text'] = full_text
    
    return sections

def extract_keywords_from_text(text, num_keywords=15):
    """استخراج کلمات کلیدی از متن"""
    
    # کلمات کلیدی مخصوص Space Biology
    space_biology_terms = [
        # فضایی
        'microgravity', 'spaceflight', 'space', 'weightlessness', 'gravity',
        'astronaut', 'cosmonaut', 'mission', 'ISS', 'station', 'orbit', 'flight',
        'spacecraft', 'shuttle', 'rocket', 'launch', 'landing',
        
        # زیستی
        'cell', 'cellular', 'tissue', 'organ', 'organism', 'biological',
        'physiological', 'anatomy', 'physiology', 'metabolism', 'homeostasis',
        
        # سیستم‌های بدن
        'bone', 'muscle', 'skeletal', 'muscular', 'cardiovascular', 'cardiac',
        'immune', 'nervous', 'respiratory', 'digestive', 'endocrine',
        'blood', 'plasma', 'serum', 'heart', 'lung', 'kidney', 'liver',
        
        # مولکولی
        'protein', 'gene', 'DNA', 'RNA', 'enzyme', 'hormone', 'receptor',
        'molecular', 'genetic', 'genomic', 'transcription', 'expression',
        
        # تحقیقاتی
        'experiment', 'study', 'research', 'analysis', 'test', 'measurement',
        'data', 'result', 'finding', 'observation', 'investigation',
        
        # حیوانات آزمایشگاهی
        'mice', 'mouse', 'rat', 'rodent', 'animal', 'model',
        
        # اثرات فضا
        'radiation', 'cosmic', 'stress', 'adaptation', 'atrophy', 'degeneration',
        'osteoporosis', 'muscle wasting', 'bone loss', 'calcium'
    ]
    
    text_lower = text.lower()
    keyword_counts = {}
    
    # شمارش کلمات
    for term in space_biology_terms:
        count = text_lower.count(term)
        if count >= 2:  # حداقل 2 بار تکرار شده باشه
            keyword_counts[term] = count
    
    # مرتب‌سازی بر اساس تعداد
    sorted_keywords = sorted(keyword_counts.items(), key=lambda x: x[1], reverse=True)
    
    return [kw[0] for kw in sorted_keywords[:num_keywords]]

def process_single_article(article_id, title, url):
    """پردازش یک مقاله"""
    
    print(f"\n🔄 [{article_id:03d}] {title[:60]}...")
    
    # بررسی اینکه قبلاً دانلود شده یا نه
    html_path = f"data/raw_html/article_{article_id:03d}.html"
    
    if os.path.exists(html_path):
        print(f"   ✅ HTML از قبل موجود")
    else:
        # دانلود HTML
        html_path, status = download_html_from_pmc(url, article_id)
        
        if not html_path:
            print(f"   ❌ دانلود ناموفق: {status}")
            return None
        
        print(f"   ✅ HTML دانلود شد")
    
    # خواندن HTML و استخراج متن
    try:
        with open(html_path, 'r', encoding='utf-8') as f:
            html_content = f.read()
        
        print(f"   🔍 استخراج محتوا...")
        sections = extract_sections_from_html(html_content)
        
        # استخراج کلمات کلیدی
        keywords = extract_keywords_from_text(sections['full_text'])
        
        # آماده‌سازی داده نهایی
        article_data = {
            'article_id': article_id,
            'original_title': title,
            'extracted_title': sections['title'] or title,
            'authors': sections['authors'],
            'url': url,
            'abstract': sections['abstract'][:3000] if sections['abstract'] else "",
            'introduction': sections['introduction'][:2000] if sections['introduction'] else "",
            'methods': sections['methods'][:2000] if sections['methods'] else "",
            'results': sections['results'][:3000] if sections['results'] else "",
            'discussion': sections['discussion'][:2000] if sections['discussion'] else "",
            'conclusion': sections['conclusion'][:1500] if sections['conclusion'] else "",
            'references': sections['references'][:1000] if sections['references'] else "",
            'full_text': sections['full_text'][:20000],  # محدود کردن طول
            'word_count': len(sections['full_text'].split()),
            'keywords': ', '.join(keywords),
            'keywords_count': len(keywords),
            'processing_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'status': 'success'
        }
        
        print(f"   ✅ پردازش موفق ({article_data['word_count']} کلمه، {len(keywords)} کلمه کلیدی)")
        return article_data
        
    except Exception as e:
        print(f"   ❌ خطا در پردازش: {str(e)}")
        return {
            'article_id': article_id,
            'original_title': title,
            'url': url,
            'status': f'error: {str(e)}',
            'processing_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

def process_all_articles(csv_path):
    """پردازش همه مقالات"""
    
    setup_directories()
    
    # خواندن CSV اصلی
    print("📖 خواندن فایل CSV...")
    df = pd.read_csv(csv_path)
    print(f"📁 تعداد مقالات: {len(df)}")
    
    all_articles = []
    failed_articles = []
    
    start_time = datetime.now()
    
    for idx, row in df.iterrows():
        article_id = idx + 1
        title = row['Title']
        url = row['Link']
        
        # پردازش مقاله
        result = process_single_article(article_id, title, url)
        
        if result:
            if result.get('status') == 'success':
                all_articles.append(result)
            else:
                failed_articles.append(result)
        else:
            failed_articles.append({
                'article_id': article_id,
                'original_title': title,
                'url': url,
                'status': 'completely_failed'
            })
        
        # ذخیره موقت هر 20 مقاله
        if len(all_articles) % 20 == 0 and len(all_articles) > 0:
            temp_df = pd.DataFrame(all_articles)
            temp_df.to_csv('data/processed_data/temp_progress.csv', index=False)
            print(f"💾 ذخیره موقت: {len(all_articles)} مقاله موفق")
        
        # استراحت کوتاه
        time.sleep(1.5)
        
        # نمایش پیشرفت
        if article_id % 50 == 0:
            elapsed = datetime.now() - start_time
            print(f"\n📊 پیشرفت: {article_id}/608 ({article_id/608*100:.1f}%)")
            print(f"⏱️  زمان سپری شده: {elapsed}")
            print(f"✅ موفق: {len(all_articles)}")
            print(f"❌ ناموفق: {len(failed_articles)}")
    
    # ذخیره نهایی
    print(f"\n🏁 پردازش تمام شد!")
    
    if all_articles:
        # ساخت DataFrame نهایی
        final_df = pd.DataFrame(all_articles)
        
        # مرتب‌سازی بر اساس article_id
        final_df = final_df.sort_values('article_id').reset_index(drop=True)
        
        # ذخیره CSV نهایی
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        final_csv_path = f'data/processed_data/space_biology_articles_{timestamp}.csv'
        final_df.to_csv(final_csv_path, index=False)
        
        # ذخیره JSON هم
        final_json_path = f'data/processed_data/space_biology_articles_{timestamp}.json'
        final_df.to_json(final_json_path, orient='records', ensure_ascii=False, indent=2)
        
        # آمار نهایی
        total_time = datetime.now() - start_time
        
        print(f"\n📊 آمار نهایی:")
        print(f"✅ مقالات موفق: {len(all_articles)}")
        print(f"❌ مقالات ناموفق: {len(failed_articles)}")
        print(f"📄 کل مقالات: {len(df)}")
        print(f"⏱️  کل زمان: {total_time}")
        print(f"💾 فایل CSV: {final_csv_path}")
        print(f"💾 فایل JSON: {final_json_path}")
        
        # نمایش نمونه از داده‌ها
        print(f"\n📋 نمونه ستون‌های CSV:")
        for col in final_df.columns:
            print(f"   - {col}")
        
        # ذخیره مقالات ناموفق
        if failed_articles:
            failed_df = pd.DataFrame(failed_articles)
            failed_csv_path = f'data/processed_data/failed_articles_{timestamp}.csv'
            failed_df.to_csv(failed_csv_path, index=False)
            print(f"❌ مقالات ناموفق: {failed_csv_path}")
        
        return final_df
    
    else:
        print("❌ هیچ مقاله‌ای پردازش نشد!")
        return None

if __name__ == "__main__":
    print("🚀 Space Biology Knowledge Engine - Data Processor")
    print("=" * 50)
    
    # دریافت نام فایل CSV
    csv_file = input("📁 نام فایل CSV خودت رو وارد کن: ")
    
    if not os.path.exists(csv_file):
        print(f"❌ فایل {csv_file} پیدا نشد!")
        print("مطمئن شو که فایل در همین مسیر هست.")
    else:
        print(f"✅ فایل {csv_file} پیدا شد")
        print("\n🏃‍♂️ شروع پردازش...")
        
        # شروع پردازش
        result_df = process_all_articles(csv_file)
        
        if result_df is not None:
            print("\n🎉 پردازش با موفقیت تمام شد!")
            print(f"📊 داده‌های نهایی آماده استفاده در پوشه data/processed_data")
        else:
            print("\n😞 پردازش ناموفق بود!")
        
        print("\n" + "=" * 50)

🚀 Space Biology Knowledge Engine - Data Processor
✅ فایل articles.csv پیدا شد

🏃‍♂️ شروع پردازش...
✅ پوشه‌ها ساخته شدند
📖 خواندن فایل CSV...
📁 تعداد مقالات: 607

🔄 [001] Mice in Bion-M 1 space mission: training and selection...
   📡 تلاش دانلود 1/3
   ✅ HTML دانلود شد
   🔍 استخراج محتوا...
   ✅ پردازش موفق (11341 کلمه، 15 کلمه کلیدی)

🔄 [002] Microgravity induces pelvic bone loss through osteoclastic a...
   📡 تلاش دانلود 1/3
   ✅ HTML دانلود شد
   🔍 استخراج محتوا...
   ✅ پردازش موفق (12472 کلمه، 15 کلمه کلیدی)

🔄 [003] Stem Cell Health and Tissue Regeneration in Microgravity...
   📡 تلاش دانلود 1/3
   ✅ HTML دانلود شد
   🔍 استخراج محتوا...
   ✅ پردازش موفق (11262 کلمه، 15 کلمه کلیدی)

🔄 [004] Microgravity Reduces the Differentiation and Regenerative Po...
   📡 تلاش دانلود 1/3
   ✅ HTML دانلود شد
   🔍 استخراج محتوا...
   ✅ پردازش موفق (7790 کلمه، 15 کلمه کلیدی)

🔄 [005] Microgravity validation of a novel system for RNA isolation ...
   📡 تلاش دانلود 1/3
   ✅ HTML دانلود شد
   🔍 استخراج

TypeError: to_json() got an unexpected keyword argument 'ensure_ascii'

In [1]:
import pandas as pd

# بارگذاری CSV
df = pd.read_csv('data/processed_data/space_biology_articles_20250920_102813.csv')

print(f"📊 آمار CSV:")
print(f"تعداد مقالات: {len(df)}")
print(f"مقالات موفق: {len(df[df['status'] == 'success'])}")
print(f"متوسط تعداد کلمات: {df['word_count'].mean():.0f}")
print(f"متوسط کلمات کلیدی: {df['keywords_count'].mean():.1f}")

# نمایش چند مقاله نمونه
print(f"\nنمونه مقالات:")
for i in range(3):
    print(f"\n--- مقاله {i+1} ---")
    print(f"عنوان: {df.iloc[i]['extracted_title']}")
    print(f"کلمات کلیدی: {df.iloc[i]['keywords'][:100]}...")
    print(f"تعداد کلمات: {df.iloc[i]['word_count']}")

📊 آمار CSV:
تعداد مقالات: 607
مقالات موفق: 607
متوسط تعداد کلمات: 9427
متوسط کلمات کلیدی: 14.7

نمونه مقالات:

--- مقاله 1 ---
عنوان: Mice in Bion-M 1 Space Mission: Training and Selection
کلمات کلیدی: mice, flight, experiment, rat, animal, space, test, data, mission, mouse, research, gravity, spacefl...
تعداد کلمات: 11341

--- مقاله 2 ---
عنوان: Microgravity Induces Pelvic Bone Loss through Osteoclastic Activity, Osteocytic Osteolysis, and Osteoblastic Cell Cycle Inhibition by CDKN1a/p21
کلمات کلیدی: bone, flight, rat, space, cell, gene, spaceflight, expression, gravity, microgravity, analysis, prot...
تعداد کلمات: 12472

--- مقاله 3 ---
عنوان: Microgravity and Cellular Biology: Insights into Cellular Responses and Implications for Human Health
کلمات کلیدی: cell, gravity, microgravity, rat, gene, cellular, space, immune, tissue, expression, research, findi...
تعداد کلمات: 11262


In [2]:
import pandas as pd
import os

# پیدا کردن فایل CSV
processed_files = [f for f in os.listdir('data/processed_data/') if f.startswith('space_biology_articles_')]
if processed_files:
    latest_file = max(processed_files)
    print(f"فایل CSV: {latest_file}")
    
    df = pd.read_csv(f'data/processed_data/{latest_file}')
    print(f"تعداد مقالات: {len(df)}")
    print(f"ستون‌ها: {list(df.columns)}")
    print(f"مقالات موفق: {len(df[df['status'] == 'success'])}")
    
    # نمایش نمونه
    sample = df[df['status'] == 'success'].iloc[0]
    print(f"\nنمونه مقاله:")
    print(f"عنوان: {sample['extracted_title']}")
    print(f"تعداد کلمات: {sample['word_count']}")
    print(f"کلمات کلیدی: {sample['keywords'][:100]}...")
else:
    print("❌ فایل CSV یافت نشد!")

فایل CSV: space_biology_articles_20250920_102813.csv
تعداد مقالات: 607
ستون‌ها: ['article_id', 'original_title', 'extracted_title', 'authors', 'url', 'abstract', 'introduction', 'methods', 'results', 'discussion', 'conclusion', 'references', 'full_text', 'word_count', 'keywords', 'keywords_count', 'processing_date', 'status']
مقالات موفق: 607

نمونه مقاله:
عنوان: Mice in Bion-M 1 Space Mission: Training and Selection
تعداد کلمات: 11341
کلمات کلیدی: mice, flight, experiment, rat, animal, space, test, data, mission, mouse, research, gravity, spacefl...
